In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from dm_control import suite
from env_humanoid import HumanoidStateDataset
import os
import re
import glob

CKPT_DIR = '/scorpio/home/yubei-stu-2/tcond/ckpt/'


# ============ CEM planner ============
@torch.no_grad()
def cem_plan(predict_endpoint, obs_norm, ds, reward_fn,
             horizon, act_dim, prev_mean=None,
             n_iter=3, n_samples=256, n_elite=32,
             act_low=-1.0, act_high=1.0, init_std=0.5, device='cuda'):
    """
    predict_endpoint: (obs_norm[B,S], acts_norm[B,K,A]) -> pred_norm[B,S]
    obs_norm: [S] normalized current obs
    reward_fn: (raw_state[B,S]) -> reward[B]
    prev_mean: [K,A] warm start from previous plan; None on first step
    Returns: full plan [K,A] in raw action space. Caller executes plan[0].
    """
    mean = prev_mean if prev_mean is not None else torch.zeros(horizon, act_dim, device=device)
    std = torch.full((horizon, act_dim), init_std, device=device)
    obs_b = obs_norm.unsqueeze(0).expand(n_samples, -1)

    for _ in range(n_iter):
        eps = torch.randn(n_samples, horizon, act_dim, device=device)
        acts_raw = (mean + std * eps).clamp(act_low, act_high)         # [N, K, A]
        acts_norm = ds.normalize_action(acts_raw)

        pred_norm = predict_endpoint(obs_b, acts_norm)                  # [N, S]
        pred_raw = ds.unnormalize_state(pred_norm)
        rewards = reward_fn(pred_raw)                                    # [N]

        elite = acts_raw[rewards.topk(n_elite).indices]                  # [E, K, A]
        mean = elite.mean(dim=0)
        std = elite.std(dim=0).clamp(min=0.05)                           # avoid collapse

    return mean


# ============ episode rollout ============
def run_episode(env, predict_endpoint, ds, reward_fn,
                horizon=50, act_dim=6, device='cuda'):
    ts = env.reset()
    obs = env.physics.get_state()                          # <-- 改这里
    total_r, prev_mean = 0.0, None

    while not ts.last():
        obs_norm = ds.normalize_state(torch.from_numpy(obs).float().to(device))
        plan = cem_plan(predict_endpoint, obs_norm, ds, reward_fn,
                        horizon=horizon, act_dim=act_dim,
                        prev_mean=prev_mean, device=device)
        action = plan[0].cpu().numpy()
        ts = env.step(action)
        obs = env.physics.get_state()                      # <-- 和这里
        total_r += ts.reward or 0.0                        # 顺手防 None
        prev_mean = torch.cat([plan[1:], torch.zeros(1, act_dim, device=device)], dim=0)

    return total_r


# ============ world-model wrappers ============
def make_dpwm_fn(model):
    """DPWM / ADM: input already matches (obs_norm, acts_norm) -> pred_norm."""
    def fn(obs, acts):
        out = model(obs, acts)
        return out[0] if isinstance(out, tuple) else out
    return fn


def make_dreamer_fn(model, ds, state_dim, pos_dim=9):
    """Wrap Dreamer's propiro_pred to the (obs_norm, acts_norm) -> pred_norm interface."""
    def fn(obs_norm, acts_norm):
        B, K, _ = acts_norm.shape
        obs_raw = ds.unnormalize_state(obs_norm)
        acts_raw = ds.unnormalize_action(acts_norm)
        states = torch.zeros(B, K + 1, state_dim, device=obs_norm.device)
        states[:, 0] = obs_raw
        # action[t] leads into state[t]; index 0 is the conditioning-state placeholder.
        actions = torch.cat([
            torch.zeros(B, 1, acts_raw.shape[-1], device=acts_norm.device),
            acts_raw,
        ], dim=1)
        is_first = torch.zeros(B, K + 1, 1, device=obs_norm.device)
        is_first[:, 0] = 1.0
        if model._config.nq != 0:
            targets = {"position": states[..., :pos_dim], "velocity": states[..., pos_dim:]}
        else:
            targets = {"state": states}
        eval_data = {
            "targets": targets,
            "actions": actions,
            "is_first": is_first,
        }
        pred, _ = model.propiro_pred(eval_data, condition_steps=1)       # [B, K, S] raw
        return ds.normalize_state(pred[:, -1, :])
    return fn

def make_dmc_reward(domain, task, device):
    """
    独立 sim env,不污染真 env 的 physics。
    输入 raw state [B, S],输出 reward [B]。
    """
    sim_env = suite.load(domain, task)
    physics = sim_env.physics
    task_obj = sim_env.task

    def fn(state_raw):
        state_np = state_raw.detach().cpu().numpy().astype(np.float64)
        r = np.empty(state_np.shape[0], dtype=np.float32)
        for i, s in enumerate(state_np):
            with physics.reset_context():
                physics.set_state(s)
            physics.forward()
            r[i] = task_obj.get_reward(physics)
        return torch.from_numpy(r).to(device).to(state_raw.dtype)
    return fn


# ============ reward functions (dm_control) ============
def cheetah_run_reward(state_raw):
    # state = [qpos(9), qvel(9)]; qvel[0] is root x-velocity; target speed = 10
    vel = state_raw[:, 9]
    return (vel / 10.0).clamp(0.0, 1.0)


def humanoid_walk_reward(state_raw):
    # Rough proxy: encourage forward CoM velocity + upright torso height (qpos[2]).
    # For a faithful reward, use the physics-based version below.
    vel_x = state_raw[:, 28]                     # first qvel component
    height = state_raw[:, 2]                     # torso z in qpos
    upright = (height / 1.4).clamp(0.0, 1.0)
    move = (vel_x / 1.0).clamp(0.0, 1.0)
    return upright * (0.2 + 0.8 * move)

def walker_walk_reward(state_raw):
    rootz = state_raw[:, 0]     # walker qpos: [rootz, rootx, rooty, ...]
    vel_x = state_raw[:, 10]    # qvel[1] = rootx velocity
    stand = (1.0 + rootz).clamp(0.0, 1.0)   # rootz=0 站立，rootz=-1 倒地
    move  = (vel_x / 1.0).clamp(0.0, 1.0)
    return stand * (0.2 + 0.8 * move)

# Optional: exact dm_control reward via physics.set_state — slow but ground truth.
# Loops over batch, so only enable for small n_samples or as a sanity check.
# def dmc_true_reward(state_raw, task, physics):
#     r = np.empty(state_raw.shape[0], dtype=np.float32)
#     for i, s in enumerate(state_raw.cpu().numpy()):
#         physics.set_state(s); physics.forward()
#         r[i] = task.get_reward(physics)
#     return torch.from_numpy(r).to(state_raw.device)

/scorpio/home/yubei-stu-2/miniconda3/envs/smallworld/lib/python3.10/site-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


In [2]:
# todo
device = torch.device('cuda:0')
task = 'humanoid_walk'
eval_data_path = '/scorpio/home/yubei-stu-2/tcond/data/humanoid_walk_state_random_eval.npz'
ds = HumanoidStateDataset(eval_data_path, normalize=False)
models, histories = {}, {}
# 从训练脚本里复制这些常量，保持一致
ACTION_EMBEDDING = False
OBS_HIDDEN_DIM = 256
DYN_HIDDEN_DIM = 512
if task == 'humanoid_walk':
    OBS_DIM        = 55
    OUTPUT_DIM     = 55
    ACTION_NUM     = 21
elif task == 'cheetah_run':
    OBS_DIM        = 18 # 55
    OUTPUT_DIM     = 18 # 55
    ACTION_NUM     = 6 # 21  
elif task == 'hopper_hop':
    OBS_DIM        = 14
    OUTPUT_DIM     = 14
    ACTION_NUM     = 4
elif task == 'walker_walk':
    OBS_DIM        = 18
    OUTPUT_DIM     = 18
    ACTION_NUM     = 6
else:
    pass

dataset normalize:  False
load data from /scorpio/home/yubei-stu-2/tcond/data/humanoid_walk_state_random_eval.npz


HumanoidStateDataset: 50 trajs x 1000 steps
  state_dim=55, action_dim=21


In [ ]:
from load_dreamer import load_dreamer
dreamer_task = 'humanoid'
name = f'humanoid-random-llr-tcond-norm-sch2'
ckpt_path = f'/scorpio/home/yubei-stu-2/dreamerv3_torch_ver/logdir/{name}/best.pt'
model, env, _ = load_dreamer(f'{dreamer_task}', ckpt_path, device)
model._wm.to(device)
model._wm.eval()
models['dreamer-'+name] = model
print('dreamer-'+name)
paper_name = 'dreamer-'+name

encoder shape  {'state': (55,)}
Encoder CNN shapes: {}
Encoder MLP shapes: {'state': (55,)}
21
Decoder CNN shapes: {}
Decoder MLP shapes: {'state': (55,)}
opitmizer clip 1000
Optimizer model_opt has 16540983 variables.
Missing keys: []
Unexpected keys: []
dreamer-humanoid-random-llr-tcond-norm-sch2


## score compute

In [5]:
dpwm_model = models[paper_name]
dpwm_model.eval()

# 建 CEM 的 predict_fn
# predict_fn = make_dpwm_fn(dpwm_model)
predict_fn = make_dreamer_fn(dpwm_model, ds, 55, pos_dim=28)

# 建环境。注意你训模型时用 walker_walk，dm_control 里对应 domain='walker' task='walk'
if 'cheetah' in paper_name:
    env = suite.load('cheetah', 'run');    act_dim = 6;  reward_fn = cheetah_run_reward
elif 'walker' in paper_name:
    env = suite.load('walker',  'run');   act_dim = 6;  reward_fn = walker_walk_reward
elif 'humanoid' in paper_name:
    env = suite.load('humanoid','walk');   act_dim = 21; reward_fn = make_dmc_reward('humanoid', 'walk', device) # humanoid_walk_reward

rewards = []
for seed in range(5):
    np.random.seed(seed); torch.manual_seed(seed)
    r = run_episode(env, predict_fn, ds, reward_fn,
                    horizon=50, act_dim=act_dim, device=device)
    rewards.append(r); print(f"seed {seed}: {r:.1f}")
print(f"{paper_name}: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}")

seed 0: 0.9
seed 1: 0.7
seed 2: 0.4
seed 3: 0.4
seed 4: 1.9
dreamer-humanoid-random-llr-tcond-norm-sch2: 0.9 ± 0.6


seed 0: 114.7
seed 1: 151.4
seed 2: 146.4
seed 3: 136.9
seed 4: 145.1
cheetah_run_rope_film_100K_rand_weyl_norm_logsampleK_500step: 138.9 ± 13.0

seed 0: 89.9
seed 1: 95.1
seed 2: 95.9
seed 3: 94.5
seed 4: 82.7
cheetah_run_arm_100K_rand_weyl_norm_slr_2800step: 91.6 ± 4.9

seed 0: 7.5
seed 1: 18.0
seed 2: 8.0
seed 3: 8.8
seed 4: 7.9
cheetah_run_arm_3K_rand_weyl_norm_slr: 10.1 ± 4.0

2.8810540584858937
4.994391719502692
3.9339036848451494
1.3138257126645427
2.031833428473826
random: 3.0 ± 1.3

In [13]:
model.eval()

predict_fn_dreamer = make_dreamer_fn(
    model,
    ds,
    state_dim=18,  # cheetah: 9 qpos + 9 qvel
    pos_dim=9,
)

batch = ds.sample_batch(4, K=16)
print(batch['actions'].shape)
obs = batch["obs"].to(device)
acts = batch["actions"].to(device)
target = batch["target"].to(device)

with torch.no_grad():
    pred = predict_fn_dreamer(obs, acts)
    print(pred.shape)

print("dreamer pred normalized:", pred[0])
print("target normalized:", target[0])
print("normalized MSE:", F.mse_loss(pred, target).item())

pred_raw = ds.unnormalize_state(pred)
target_raw = ds.unnormalize_state(target)
print("raw MSE:", F.mse_loss(pred_raw, target_raw).item())

torch.Size([4, 16, 6])
torch.Size([4, 18])
dreamer pred normalized: tensor([-6.3238e-01, -7.9625e-02,  4.4797e-02, -9.8883e-02,  5.7985e-02,
         2.4606e-01, -7.8790e-02,  2.1239e-01, -3.3466e-01,  1.0905e-03,
         4.3949e-02, -1.0287e+00, -3.3572e+00,  5.9099e+00, -3.6597e+00,
         7.7201e-02,  6.4617e+00, -5.5842e+00], device='cuda:0')
target normalized: tensor([-1.6803e+00, -1.3163e-01,  3.9185e-02, -2.4393e-03,  1.2410e-01,
         2.7353e-01, -7.4622e-02,  5.3574e-02, -5.0161e-01, -1.0264e+00,
         1.7673e-01, -8.6442e-01, -3.8531e+00,  5.5164e+00, -2.3471e+00,
        -1.0513e+00,  2.0059e+00, -8.8190e-01], device='cuda:0')
normalized MSE: 2.272566556930542
raw MSE: 2.272566556930542


In [11]:
def run_episode_random(env, act_dim):
    ts = env.reset()
    total_r = 0.0
    while not ts.last():
        action = np.random.uniform(-1.0, 1.0, act_dim).astype(np.float32)
        ts = env.step(action)
        total_r += ts.reward or 0.0
    return total_r

rewards = []
for seed in range(5):
    np.random.seed(seed)
    r = run_episode_random(env, act_dim)
    print(r)
    rewards.append(r)
print(f"random: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}")

2.8810540584858937
4.994391719502692
3.9339036848451494
1.3138257126645427
2.031833428473826
random: 3.0 ± 1.3


In [ ]:
# 直接用 dataset 采一个 batch，绕过 env
batch = ds.sample_batch(4, K=50)
obs = batch['obs'].to(device)          # 已归一化
acts = batch['actions'].to(device)     # 已归一化
target = batch['target'].to(device)    # 已归一化

pred = predict_fn(obs, acts)
print("pred normalized:", pred[0])
print("target normalized:", target[0])
print("normalized MSE:", F.mse_loss(pred, target).item())

pred_raw = ds.unnormalize_state(pred)
target_raw = ds.unnormalize_state(target)
print("pred raw[0]:", pred_raw[0])
print("target raw[0]:", target_raw[0])
print("raw MSE:", F.mse_loss(pred_raw, target_raw).item())

In [ ]:
# 用真 physics 做 planner，测试 CEM + reward 本身能不能优化 walker
from dm_control import suite

def make_dmc_reward(domain, task, device):
    """
    独立 sim env,不污染真 env 的 physics。
    输入 raw state [B, S],输出 reward [B]。
    """
    sim_env = suite.load(domain, task)
    physics = sim_env.physics
    task_obj = sim_env.task

    def fn(state_raw):
        state_np = state_raw.detach().cpu().numpy().astype(np.float64)
        r = np.empty(state_np.shape[0], dtype=np.float32)
        for i, s in enumerate(state_np):
            with physics.reset_context():
                physics.set_state(s)
            physics.forward()
            r[i] = task_obj.get_reward(physics)
        return torch.from_numpy(r).to(device).to(state_raw.dtype)
    return fn

def make_true_dynamics_fn(env_domain, env_task, device):
    from dm_control import suite
    sim_env = suite.load(env_domain, env_task)
    physics = sim_env.physics
    def fn(obs_raw, acts_raw):
        B, K, _ = acts_raw.shape
        obs_np = obs_raw.detach().cpu().numpy().astype(np.float64)
        acts_np = acts_raw.detach().cpu().numpy().astype(np.float64)
        endpoints = np.empty((B, obs_np.shape[-1]), dtype=np.float32)
        for b in range(B):
            with physics.reset_context():
                physics.set_state(obs_np[b])
            for k in range(K):
                physics.set_control(acts_np[b, k])
                physics.step()
            endpoints[b] = physics.get_state()
        return torch.from_numpy(endpoints).to(device).to(obs_raw.dtype)
    return fn

# 需要一个用 raw obs 的 CEM 变体（不做归一化）
@torch.no_grad()
def cem_plan_oracle(oracle_fn, obs_raw, reward_fn,
                    horizon, act_dim, prev_mean=None,
                    n_iter=3, n_samples=256, n_elite=32,
                    init_std=0.5, device='cuda'):
    mean = prev_mean if prev_mean is not None else torch.zeros(horizon, act_dim, device=device)
    std = torch.full((horizon, act_dim), init_std, device=device)
    obs_b = obs_raw.unsqueeze(0).expand(n_samples, -1)
    for _ in range(n_iter):
        eps = torch.randn(n_samples, horizon, act_dim, device=device)
        acts_raw = (mean + std * eps).clamp(-1.0, 1.0)
        pred_raw = oracle_fn(obs_b, acts_raw)
        rewards = reward_fn(pred_raw)
        elite = acts_raw[rewards.topk(n_elite).indices]
        mean = elite.mean(dim=0)
        std = elite.std(dim=0).clamp(min=0.05)
    return mean

def run_episode_oracle(env, oracle_fn, reward_fn, horizon=50, act_dim=6, device='cuda'):
    ts = env.reset()
    obs = env.physics.get_state()
    total_r, prev_mean = 0.0, None
    while not ts.last():
        obs_t = torch.from_numpy(obs).float().to(device)
        plan = cem_plan_oracle(oracle_fn, obs_t, reward_fn, horizon, act_dim, prev_mean, device=device)
        action = plan[0].cpu().numpy()
        ts = env.step(action)
        obs = env.physics.get_state()
        total_r += ts.reward or 0.0
        prev_mean = torch.cat([plan[1:], torch.zeros(1, act_dim, device=device)], dim=0)
    return total_r

# 用真 reward，避免 proxy 干扰
reward_fn_true = make_dmc_reward('walker', 'run', device)
oracle_fn = make_true_dynamics_fn('walker', 'run', device)

# 只跑一个 seed，会慢一点（大概 5-10 分钟）
np.random.seed(0); torch.manual_seed(0)
r_oracle = run_episode_oracle(env, oracle_fn, reward_fn_true, horizon=50, act_dim=6, device=device)
print("Oracle (true dyn + true reward):", r_oracle)